# 6 · Flight Dynamics

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Estimates the vertical free-flight trajectory of each 3D-printed propeller after release at a fixed RPM. The motion is a coupled 1-DOF system in height, vertical speed and spin — m·dV/dt = T(V, ω) − m·g − D(V), I·dω/dt = −Q(V, ω), dh/dt = V — integrated with RK45, with thrust and torque bilinearly interpolated from the QProp sweep surfaces. The motor is disconnected at release, so the aerodynamic torque decays the spin throughout the flight. During descent (V < 0) the propeller enters the vortex-ring state that QProp cannot model, so thrust is conservatively set to zero there. The body-drag term D(V) uses the computed axial frontal area of the bluff structure QProp does not model (ring rim + hub + blades edge-on) with Cd = 1.1; the blades' own aerodynamic drag is already inside QProp's T(V, ω). The full flow repeats across the launch-RPM grid and for the 10 recovered validation propellers.

**Physics inputs:** `csv/01_geometry.csv`, `csv/02_stl_volumes.csv`, `csv/03_mass_inertia.csv`, `csv/05_qprop_results.csv`, `csv/05_qprop_sweep.csv.gz` (and the `val_` equivalents for the validation subset)

**Physics outputs:** `csv/06_flight_dynamics_<rpm>rpm.csv` (one per launch RPM: liftoff flag, static thrust/torque/power, T/W, peak height, flight time, hover time, speeds, impact RPM), the reference `csv/06_flight_dynamics.csv` at the reference launch RPM, and the validation-subset equivalents `csv/val_06_flight_dynamics_<rpm>rpm.csv`, `csv/val_06_flight_dynamics.csv`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — input loading and gating, drag-area model, performance surfaces, equations of motion, per-config simulation, batch driver
4. Main code — input loading, inertia and drag areas, surfaces, batch simulation, export, validation subset, validation checks

> **Dead-code audit (2026-07-04):** removed the unused `POSTPROCESS_N`, `SWEEP_V_MIN`, `SWEEP_V_MAX` and `SWEEP_RPM_MIN` configuration assignments (never read).


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm

BASE_DIR = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pipeline_config as cfg
from utils import measurements

## 2. Configuration

In [ ]:
CSV_DIR = BASE_DIR / 'csv'

GEOMETRY_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['geometry']
MASS_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['mass_inertia']
STL_VOLUMES_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['stl_volumes']
SWEEP_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['qprop_sweep']
QPROP_RESULTS_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['qprop_results']
FLIGHT_DYNAMICS_CSV_PATH = CSV_DIR / cfg.CSV_NAMES['flight_dynamics']

RPM_LAUNCH = cfg.LAUNCH_RPM
INITIAL_HEIGHT_M = cfg.INITIAL_HEIGHT_M
INITIAL_VERTICAL_VELOCITY_M_S = cfg.INITIAL_VERTICAL_VELOCITY_M_S

GRAVITY_M_S2 = cfg.GRAVITY_M_S2
AIR_DENSITY_KG_M3 = cfg.AIR_DENSITY_KG_M3
BODY_DRAG_COEFFICIENT = cfg.BODY_DRAG_COEFFICIENT

TIME_LIMIT_S = cfg.TIME_LIMIT_S
ODE_METHOD = cfg.ODE_METHOD
ODE_RTOL = cfg.ODE_RTOL
ODE_ATOL = cfg.ODE_ATOL
ROUND_DECIMALS = cfg.ROUND_DECIMALS

SWEEP_RPM_MAX = cfg.SWEEP_RPM_MAX

OUTPUT_COLUMNS = cfg.NB6_OUTPUT_COLUMNS
LAUNCH_RPMS = cfg.LAUNCH_RPMS

INNER_STATION_RADIUS_MM = cfg.INNER_ANCHOR_RADIUS_MM
HUB_RADIUS_MM = cfg.HUB_STATION_RADIUS_MM

print(f'Launch RPM : {RPM_LAUNCH}  (reference for NB7)')
print(f'Launch RPMs: {LAUNCH_RPMS}')
print(f'Sweep file : {SWEEP_CSV_PATH.name}')

## 3. Function Definitions

- **load_sweep_surfaces(sweep_csv_path)** — loads the QProp sweep, keeps only the rows that passed the plausibility gate, and adds the angular speed column. Returns the raw and the filtered sweep tables.
- **check_sweep_rpm_coverage(sweep_df, launch_rpms)** — the sweep RPM-coverage guard: the bilinear lookup clamps ω to the grid edge, so a launch RPM above the sweep ceiling would silently under-predict thrust by up to (rpm/ceiling)². This function refuses to run when the requested launch RPMs exceed the sweep coverage and returns the detected ceiling.
- **build_base_table(geometry_df, mass_df, stl_ok_df, qprop_ok_df)** — merges geometry, mass/inertia and the QProp flags into one per-config table; cross-checks `stl_ok` against the authoritative NB2 flag, excludes sentinel-mass placeholder rows, and requires `stl_ok` for `qprop_ok`.
- **add_inertia_and_drag_areas(base_df)** — adds the spin-axis inertia (preferring the validation-corrected `izz_regressed`, falling back to the raw STL `izz`) and the axial body-drag frontal area: ring annulus + hub disc + blades edge-on (thickness × span), computed per config from its geometry. The blade planform area is also computed for traceability but is not used for drag, since the blade aerodynamic drag is already inside QProp's T(V, ω).
- **build_performance_surface(table)** — pivots one config's sweep rows into dense T and Q grids over (V, ω), filling holes with the nearest available sweep point.
- **build_all_surfaces(sweep_df)** — pre-builds the performance surfaces for every config in the sweep.
- **surface_omega_max(surface)** — the highest rotation speed present in one propeller's QProp data.
- **interp_surface(surface, v_query, omega_query)** — bilinearly interpolates thrust and torque at any velocity and rotation speed between grid points, clamped to the grid edges.
- **ground_hit(t, state)** — the ODE stop condition: fires when the propeller falls back through its launch height (terminal, downward direction only).
- **make_eom(mass_kg, inertia_kg_m2, frontal_area_drag_m2, surface)** — builds the equations-of-motion function for one propeller: thrust minus weight minus body drag for the vertical acceleration, aerodynamic torque decay for the spin, T = 0 during descent (vortex-ring state), spin clamped non-negative.
- **simulate_config(row, surface, rpm_launch, omega_launch)** — simulates one propeller at one launch RPM: skips configs whose own sweep does not reach the launch RPM (no extrapolation), computes the static hover point and the liftoff flag, integrates the ODE for liftoff-capable configs, and returns peak height, flight time, hover time, maximum and impact speed and impact RPM.
- **make_skip_record(row, launch_rpm, output_columns)** — the placeholder record for configs without QProp data.
- **simulate_all_rpms(run_df, skip_df, surfaces, launch_rpms, output_columns, label)** — runs the batch across the full launch-RPM grid and returns one output table per RPM.
- **chk(cond, msg)** — pass/fail assertion helper used by the validation checks.


In [ ]:
def load_sweep_surfaces(sweep_csv_path):

    sweep_raw = pd.read_csv(sweep_csv_path)
    sweep_df = sweep_raw[sweep_raw['qprop_ok']].copy()
    sweep_df = sweep_df[['config_id', 'V', 'rpm', 'T', 'Q', 'Pshaft']].copy()
    sweep_df = sweep_df.dropna(subset=['T', 'Q'])
    sweep_df['omega'] = sweep_df['rpm'] * 2.0 * math.pi / 60.0

    return sweep_raw, sweep_df


def check_sweep_rpm_coverage(sweep_df, launch_rpms):

    sweep_rpm_ceiling = float(sweep_df['rpm'].max())
    requested_max_rpm = max(launch_rpms)
    if requested_max_rpm > sweep_rpm_ceiling + 1.0:
        raise RuntimeError(f'Sweep covers RPM up to {sweep_rpm_ceiling:.0f}, but pipeline_config requests launch RPM up to {requested_max_rpm}. Re-run NB5 (QProp sweep) with QPROP_OVERWRITE_RUNS=True so the sweep spans the full {cfg.RPM_MIN}-{cfg.RPM_MAX} rpm grid before running this notebook. Silent omega-clamping is disabled to keep results thesis-defendable.')

    return sweep_rpm_ceiling


def build_base_table(geometry_df, mass_df, stl_ok_df, qprop_ok_df):

    nb2_stl_ok = stl_ok_df.set_index('config_id')['stl_ok'].astype(bool)

    if 'izz_regressed' in mass_df.columns:
        mass_columns = mass_df[['config_id', 'mass_kg', 'izz', 'izz_regressed', 'frontal_area_m2', 'stl_ok']]
    else:
        mass_columns = mass_df[['config_id', 'mass_kg', 'izz', 'frontal_area_m2', 'stl_ok']]
    base_df = geometry_df.merge(mass_columns, on='config_id', how='inner').merge(qprop_ok_df[['config_id', 'qprop_ok']], on='config_id', how='left').sort_values('config_id').reset_index(drop=True)

    base_df['qprop_ok'] = base_df['qprop_ok'].fillna(False).astype(bool)
    base_df['stl_ok_nb2'] = base_df['config_id'].map(nb2_stl_ok).fillna(False).astype(bool)
    base_df['stl_ok'] = base_df['stl_ok'] & base_df['stl_ok_nb2']

    sentinel_mass_kg = 0.000674
    base_df.loc[base_df['mass_kg'] <= sentinel_mass_kg, 'stl_ok'] = False

    base_df['qprop_ok'] = base_df['qprop_ok'] & base_df['stl_ok']

    return base_df


def add_inertia_and_drag_areas(base_df):

    if 'izz_regressed' in base_df.columns:
        base_df['inertia_kg_m2'] = base_df['izz_regressed'].fillna(base_df['izz'])
    else:
        base_df['inertia_kg_m2'] = base_df['izz']

    R_m = base_df['radius_mm'].values / 1000.0
    rt_m = base_df['ring_thickness_mm'].values / 1000.0
    hub_m = HUB_RADIUS_MM / 1000.0

    A_ring = np.pi * (R_m ** 2 - np.clip(R_m - rt_m, 0.0, None) ** 2)
    A_hub = np.pi * hub_m ** 2

    t_inner = (base_df['inner_thickness_pct'].values / 100.0) * (base_df['inner_chord_mm'].values / 1000.0)
    t_outer = (base_df['outer_thickness_pct'].values / 100.0) * (base_df['outer_chord_mm'].values / 1000.0)
    t_blade_mean = 0.5 * (t_inner + t_outer)
    span_m = np.clip(R_m - hub_m, 0.0, None)
    A_blade_edge = t_blade_mean * span_m * base_df['blade_count'].values

    base_df['frontal_area_drag_m2'] = A_ring + A_hub + A_blade_edge

    r_inner = INNER_STATION_RADIUS_MM / 1000.0 * np.ones(len(base_df))
    r_mid = base_df['mid_radial_pos'].values * R_m
    c_inner = base_df['inner_chord_mm'].values / 1000.0
    c_mid = base_df['mid_chord_mm'].values / 1000.0
    c_outer = base_df['outer_chord_mm'].values / 1000.0
    area_per_blade = 0.5 * (c_inner + c_mid) * (r_mid - r_inner) + 0.5 * (c_mid + c_outer) * (R_m - r_mid)
    base_df['blade_planform_m2'] = area_per_blade * base_df['blade_count'].values

    return base_df


def build_performance_surface(table):

    V_vals = np.sort(table['V'].unique())
    omega_vals = np.sort(table['omega'].unique())
    n_v = len(V_vals)
    n_w = len(omega_vals)
    T_grid = np.full((n_v, n_w), np.nan)
    Q_grid = np.full((n_v, n_w), np.nan)

    for row in table.itertuples(index=False):
        iv = int(np.searchsorted(V_vals, row.V, side='left'))
        iw = int(np.searchsorted(omega_vals, row.omega, side='left'))
        if iv < n_v and iw < n_w:
            T_grid[iv, iw] = row.T
            Q_grid[iv, iw] = row.Q

    for i in range(n_v):
        for j in range(n_w):
            if np.isnan(T_grid[i, j]):
                dist = np.abs(table['V'] - V_vals[i]) + np.abs(table['omega'] - omega_vals[j])
                idx = dist.idxmin()
                T_grid[i, j] = float(table.loc[idx, 'T'])
                Q_grid[i, j] = float(table.loc[idx, 'Q'])

    return V_vals, omega_vals, T_grid, Q_grid


def build_all_surfaces(sweep_df):

    surfaces = {}
    for cid, group in sweep_df.groupby('config_id'):
        surfaces[int(cid)] = build_performance_surface(group.reset_index(drop=True))

    return surfaces


def surface_omega_max(surface):

    if surface is None:
        omega_max = 0.0
    else:
        omega_max = float(surface[1][-1])

    return omega_max


def interp_surface(surface, v_query, omega_query):

    V_vals, omega_vals, T_grid, Q_grid = surface

    v = float(np.clip(v_query, V_vals[0], V_vals[-1]))
    omega = float(np.clip(omega_query, omega_vals[0], omega_vals[-1]))

    i_v = int(np.clip(np.searchsorted(V_vals, v, side='right') - 1, 0, len(V_vals) - 2))
    i_w = int(np.clip(np.searchsorted(omega_vals, omega, side='right') - 1, 0, len(omega_vals) - 2))

    v0 = V_vals[i_v]
    v1 = V_vals[i_v + 1]
    w0 = omega_vals[i_w]
    w1 = omega_vals[i_w + 1]
    if v1 > v0:
        tv = (v - v0) / (v1 - v0)
    else:
        tv = 0.0
    if w1 > w0:
        tw = (omega - w0) / (w1 - w0)
    else:
        tw = 0.0

    T = (T_grid[i_v, i_w] * (1 - tv) * (1 - tw) + T_grid[i_v + 1, i_w] * tv * (1 - tw) + T_grid[i_v, i_w + 1] * (1 - tv) * tw + T_grid[i_v + 1, i_w + 1] * tv * tw)
    Q = (Q_grid[i_v, i_w] * (1 - tv) * (1 - tw) + Q_grid[i_v + 1, i_w] * tv * (1 - tw) + Q_grid[i_v, i_w + 1] * (1 - tv) * tw + Q_grid[i_v + 1, i_w + 1] * tv * tw)

    return float(T), float(Q)


def ground_hit(t, state):

    h, V, omega = state

    return h - INITIAL_HEIGHT_M


ground_hit.terminal = True
ground_hit.direction = -1


def make_eom(mass_kg, inertia_kg_m2, frontal_area_drag_m2, surface):

    weight = mass_kg * GRAVITY_M_S2

    def eom(t, state):

        h, V, omega = state
        omega = max(omega, 0.0)

        if omega > 0.5:
            if V >= 0.0:
                T, Q = interp_surface(surface, V, omega)
                Q = max(Q, 0.0)
            else:
                T = 0.0
                Q = max(interp_surface(surface, 0.0, omega)[1], 0.0)
        else:
            T = 0.0
            Q = 0.0

        D = 0.5 * AIR_DENSITY_KG_M3 * BODY_DRAG_COEFFICIENT * frontal_area_drag_m2 * V * abs(V)

        dh_dt = V
        dV_dt = (T - weight - D) / mass_kg
        if omega > 0.5:
            domega_dt = -Q / inertia_kg_m2
        else:
            domega_dt = 0.0

        return [dh_dt, dV_dt, domega_dt]

    return eom


def make_skip_record(row, launch_rpm, output_columns):

    record = {
        'config_id': int(row['config_id']),
        'flight_ok': False,
        'can_liftoff': False,
        'rpm_launch': float(launch_rpm),
    }
    for column in ['mass_kg', 'inertia_kg_m2', 'blade_planform_m2', 'frontal_area_drag_m2']:
        value = row.get(column, np.nan)
        if pd.notna(value):
            record[column] = float(value)
        else:
            record[column] = np.nan
    for column in output_columns:
        if column not in record:
            record[column] = np.nan

    return record


def chk(cond, msg):

    global all_ok
    print(f'  [{"OK  " if cond else "FAIL"}]  {msg}')
    if not cond:
        all_ok = False

In [ ]:
def simulate_config(row, surface, rpm_launch, omega_launch):

    cid = int(row['config_id'])
    mass_kg = float(row['mass_kg'])
    inertia_kg_m2 = float(row['inertia_kg_m2'])
    blade_planform = float(row['blade_planform_m2'])
    frontal_drag = float(row['frontal_area_drag_m2'])
    weight = mass_kg * GRAVITY_M_S2

    rec = {
        'config_id': cid,
        'flight_ok': False,
        'config_covered': True,
        'can_liftoff': False,
        'rpm_launch': rpm_launch,
        'mass_kg': mass_kg,
        'inertia_kg_m2': inertia_kg_m2,
        'blade_planform_m2': blade_planform,
        'frontal_area_drag_m2': frontal_drag,
        'T_static_N': np.nan,
        'Q_static_Nm': np.nan,
        'Pshaft_static_W': np.nan,
        'T_over_W': np.nan,
        'h_max_m': np.nan,
        'flight_time_s': np.nan,
        'hover_time_s': np.nan,
        'v_max_m_s': np.nan,
        'v_impact_m_s': np.nan,
        'rpm_at_impact': np.nan,
    }

    if surface is None:

        return rec

    if omega_launch > surface_omega_max(surface) * 1.001:
        rec['config_covered'] = False

        return rec

    rec['config_covered'] = True

    T_static, Q_static = interp_surface(surface, 0.0, omega_launch)
    T_static = max(T_static, 0.0)
    Q_static = max(Q_static, 0.0)
    Pshaft_static = Q_static * omega_launch

    if weight > 0:
        thrust_to_weight = round(T_static / weight, 5)
    else:
        thrust_to_weight = np.nan
    rec.update({
        'flight_ok': True,
        'can_liftoff': bool(T_static > weight),
        'T_static_N': round(T_static, 6),
        'Q_static_Nm': round(Q_static, 7),
        'Pshaft_static_W': round(Pshaft_static, 5),
        'T_over_W': thrust_to_weight,
    })

    if not rec['can_liftoff']:
        rec.update({
            'h_max_m': 0.0,
            'flight_time_s': 0.0,
            'hover_time_s': 0.0,
            'v_max_m_s': 0.0,
            'v_impact_m_s': 0.0,
            'rpm_at_impact': rpm_launch,
        })

        return rec

    y0 = [INITIAL_HEIGHT_M, INITIAL_VERTICAL_VELOCITY_M_S, omega_launch]
    eom = make_eom(mass_kg, inertia_kg_m2, frontal_drag, surface)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            sol = solve_ivp(eom, (0.0, TIME_LIMIT_S), y0, events=ground_hit, method=ODE_METHOD, rtol=ODE_RTOL, atol=ODE_ATOL, dense_output=False)
    except Exception as exc:
        warnings.warn(f'ODE failed config {cid}: {exc}')
        rec['flight_ok'] = False

        return rec

    if not sol.success or len(sol.t) < 2:
        rec['flight_ok'] = False

        return rec

    t_arr = sol.t
    h_arr = sol.y[0]
    V_arr = sol.y[1]
    w_arr = np.maximum(sol.y[2], 0.0)

    thrust_values = []
    for vv, ww in zip(V_arr, w_arr):
        thrust_values.append(interp_surface(surface, max(float(vv), 0.0), float(ww))[0])
    T_arr = np.array(thrust_values)
    hover_mask = T_arr >= weight
    if hover_mask.any():
        hover_time_s = float(t_arr[np.where(hover_mask)[0][-1]])
    else:
        hover_time_s = 0.0

    rec.update({
        'h_max_m': round(float(np.max(h_arr)), ROUND_DECIMALS),
        'flight_time_s': round(float(t_arr[-1]), ROUND_DECIMALS),
        'hover_time_s': round(hover_time_s, ROUND_DECIMALS),
        'v_max_m_s': round(float(np.max(V_arr)), ROUND_DECIMALS),
        'v_impact_m_s': round(float(V_arr[-1]), ROUND_DECIMALS),
        'rpm_at_impact': round(float(w_arr[-1] * 60.0 / (2.0 * math.pi)), ROUND_DECIMALS),
    })

    return rec


def simulate_all_rpms(run_df, skip_df, surfaces, launch_rpms, output_columns, label='Flight sims'):

    all_flight_dfs = {}
    for launch_rpm_i in launch_rpms:
        omega_launch_i = launch_rpm_i * 2.0 * math.pi / 60.0

        records = []
        for row_index, row in tqdm(run_df.iterrows(), total=len(run_df), desc=f'{label} @ {launch_rpm_i} RPM', leave=False):
            surface = surfaces.get(int(row['config_id']))
            records.append(simulate_config(row, surface, float(launch_rpm_i), omega_launch_i))

        for row_index, row in skip_df.iterrows():
            records.append(make_skip_record(row, launch_rpm_i, output_columns))

        df_i = pd.DataFrame(records).sort_values('config_id').reset_index(drop=True)
        for column in output_columns:
            if column not in df_i.columns:
                df_i[column] = np.nan
        df_i = df_i[output_columns]
        all_flight_dfs[launch_rpm_i] = df_i

    return all_flight_dfs

## 4. Main Code

The main code loads and gates the inputs, computes the per-config inertia and drag areas, pre-builds the thrust/torque surfaces, simulates every configuration across the full launch-RPM grid, exports the per-RPM tables plus the reference file, repeats the identical flow for the validation subset, and closes with pass/fail checks.


### 4.1 Load and Validate Input Data

Loads geometry, mass/inertia, the QProp flags and the full 2-D sweep, applies the sweep RPM-coverage guard, and builds the gated per-config base table.


In [ ]:
for path in [SWEEP_CSV_PATH, GEOMETRY_CSV_PATH, MASS_CSV_PATH, STL_VOLUMES_CSV_PATH, QPROP_RESULTS_CSV_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')

geometry_df = pd.read_csv(GEOMETRY_CSV_PATH)
mass_df = pd.read_csv(MASS_CSV_PATH)
stl_ok_df = pd.read_csv(STL_VOLUMES_CSV_PATH)
qprop_ok_df = pd.read_csv(QPROP_RESULTS_CSV_PATH)

print('Loading sweep CSV (this may take a moment)...')
sweep_raw, sweep_df = load_sweep_surfaces(SWEEP_CSV_PATH)

print(f'Sweep rows total    : {len(sweep_raw):,}')
print(f'Plausible rows      : {len(sweep_df):,}')
print(f'Configs with data   : {sweep_df["config_id"].nunique()}')

SWEEP_RPM_CEILING = check_sweep_rpm_coverage(sweep_df, LAUNCH_RPMS)
print(f'Sweep RPM ceiling   : {SWEEP_RPM_CEILING:.0f} rpm')

base_df = build_base_table(geometry_df, mass_df, stl_ok_df, qprop_ok_df)

n_bad_stl = int((~base_df['stl_ok']).sum())
print(f'Base rows           : {len(base_df)}')
print(f'Bad STL (excluded)  : {n_bad_stl}')
print(f'With QProp data     : {base_df["qprop_ok"].sum()}')

### 4.2 Inertia and Axial Drag Frontal Area

The spin-axis inertia comes from `csv/03_mass_inertia.csv`, preferring the validation-corrected `izz_regressed`. The body-drag frontal area represents only the bluff structure QProp does not model. An earlier model applied flat-plate drag (Cd = 1.17) to the blade planform, which both double-counted the blade drag (already inside QProp's net thrust) and overstated the axial projected area by roughly 3×, since in axial flight the blades meet the air nearly edge-on. The corrected model uses A_frontal = ring annulus + hub frontal disc + blade edge-on area (thickness × span) with a cylinder/annulus Cd of 1.1.


In [ ]:
base_df = add_inertia_and_drag_areas(base_df)

ok_mask = base_df['stl_ok']
print('Inertia Izz [kg·m²]')
print(f'  min={base_df.loc[ok_mask, "inertia_kg_m2"].min():.3e}  mean={base_df.loc[ok_mask, "inertia_kg_m2"].mean():.3e}  max={base_df.loc[ok_mask, "inertia_kg_m2"].max():.3e}')
print()
fa_cm2 = base_df.loc[ok_mask, 'frontal_area_drag_m2'] * 1e4
bp_cm2 = base_df.loc[ok_mask, 'blade_planform_m2'] * 1e4
print('Axial frontal area [cm²]  (used for body drag D = ½ρV²·Cd_body·A_frontal)')
print(f'  ring+hub+blade-edge: min={fa_cm2.min():.1f}  mean={fa_cm2.mean():.1f}  max={fa_cm2.max():.1f}')
print('Blade planform [cm²]  (traceability only, NOT used for drag)')
print(f'  min={bp_cm2.min():.1f}  mean={bp_cm2.mean():.1f}  max={bp_cm2.max():.1f}')
print(f'  -> old planform-drag area was ~{bp_cm2.mean() / fa_cm2.mean():.1f}x the true axial frontal area')

### 4.3 QProp Performance Lookup

Pre-builds the per-config bilinear T(V, ω) / Q(V, ω) surfaces from the sweep.


In [ ]:
OMEGA_LAUNCH = RPM_LAUNCH * 2.0 * math.pi / 60.0

performance_surfaces_by_cid = build_all_surfaces(sweep_df)

test_config_id = next(iter(performance_surfaces_by_cid))
thrust_test, torque_test = interp_surface(performance_surfaces_by_cid[test_config_id], 0.0, OMEGA_LAUNCH)
print(f'Test config {test_config_id}: T(V=0, ω_launch) = {thrust_test:.4f} N,  Q = {torque_test:.5f} N·m')
print(f'Surface configs preloaded: {len(performance_surfaces_by_cid)}')

### 4.4 Batch Flight Simulation

Simulates every config with both mass/inertia and QProp data at every launch RPM in the grid. All configurations are released at the same RPM to enable direct aerodynamic comparison across the design space.


In [ ]:
run_df = base_df[base_df['qprop_ok']].copy().reset_index(drop=True)
skip_df = base_df[~base_df['qprop_ok']].copy().reset_index(drop=True)

print(f'Configs to simulate : {len(run_df)}')
print(f'Configs skipped     : {len(skip_df)}  (no QProp data)')
print(f'Launch RPMs         : {LAUNCH_RPMS}')
print()

all_flight_dfs = simulate_all_rpms(run_df, skip_df, performance_surfaces_by_cid, LAUNCH_RPMS, OUTPUT_COLUMNS)

for launch_rpm_i, df_i in all_flight_dfs.items():
    n_liftoff = int(df_i['can_liftoff'].sum())
    if n_liftoff:
        h_mean = df_i.loc[df_i['can_liftoff'], 'h_max_m'].mean()
    else:
        h_mean = float('nan')
    print(f'  {int(launch_rpm_i):5d} RPM — liftoff: {n_liftoff:4d}  h_max mean: {h_mean:.3f} m')

flight_df = all_flight_dfs[int(cfg.LAUNCH_RPM)]
print(f'\nDone. Reference flight_df = {int(cfg.LAUNCH_RPM)} RPM  ({len(flight_df)} rows)')

### 4.5 Export Results

In [ ]:
CSV_DIR.mkdir(parents=True, exist_ok=True)

for launch_rpm_i, df_i in all_flight_dfs.items():
    fname = f'06_flight_dynamics_{int(launch_rpm_i)}rpm.csv'
    df_i.to_csv(CSV_DIR / fname, index=False)
    print(f'Saved: {fname}  ({len(df_i)} rows)')

flight_df.to_csv(FLIGHT_DYNAMICS_CSV_PATH, index=False)
print(f'\nReference: {FLIGHT_DYNAMICS_CSV_PATH.name}  (= {int(cfg.LAUNCH_RPM)} RPM)')

### 4.6 Validation Subset — Recovered Geometry

Runs the identical flow for the 10 recovered validation propellers using their `val_` inputs (see NB3–NB5) and writes `csv/val_06_flight_dynamics_<rpm>rpm.csv` plus the reference `csv/val_06_flight_dynamics.csv`. NB9 reads these files to extend the flight validation to all 30 tested props.


In [ ]:
val_mass_csv = CSV_DIR / 'val_03_mass_inertia.csv'
val_stl_csv = CSV_DIR / 'val_02_stl_volumes.csv'
val_qprop_csv = CSV_DIR / 'val_05_qprop_results.csv'
val_sweep_csv = CSV_DIR / 'val_05_qprop_sweep.csv.gz'
val_geometry_path = measurements.validation_geometry_path(BASE_DIR)

val_inputs_exist = val_mass_csv.exists() and val_stl_csv.exists() and val_qprop_csv.exists() and val_sweep_csv.exists() and val_geometry_path.exists()

if val_inputs_exist:
    val_geometry_df = measurements.load_recovered_validation_geometry(BASE_DIR)
    val_mass_df = pd.read_csv(val_mass_csv)
    val_stl_ok_df = pd.read_csv(val_stl_csv)
    val_qprop_ok_df = pd.read_csv(val_qprop_csv)

    val_sweep_raw, val_sweep_df = load_sweep_surfaces(val_sweep_csv)
    val_ceiling = check_sweep_rpm_coverage(val_sweep_df, LAUNCH_RPMS)
    print(f'Validation sweep: {len(val_sweep_df):,} plausible rows, ceiling {val_ceiling:.0f} rpm')

    val_base_df = build_base_table(val_geometry_df, val_mass_df, val_stl_ok_df, val_qprop_ok_df)
    val_base_df = add_inertia_and_drag_areas(val_base_df)
    val_surfaces = build_all_surfaces(val_sweep_df)

    val_run_df = val_base_df[val_base_df['qprop_ok']].copy().reset_index(drop=True)
    val_skip_df = val_base_df[~val_base_df['qprop_ok']].copy().reset_index(drop=True)
    print(f'Validation subset: {len(val_run_df)} to simulate, {len(val_skip_df)} skipped')

    val_flight_dfs = simulate_all_rpms(val_run_df, val_skip_df, val_surfaces, LAUNCH_RPMS, OUTPUT_COLUMNS, label='Validation flight sims')

    for launch_rpm_i, df_i in val_flight_dfs.items():
        fname = f'val_06_flight_dynamics_{int(launch_rpm_i)}rpm.csv'
        df_i.to_csv(CSV_DIR / fname, index=False)
    val_flight_dfs[int(cfg.LAUNCH_RPM)].to_csv(CSV_DIR / 'val_06_flight_dynamics.csv', index=False)
    print(f'Saved val_06_flight_dynamics_*rpm.csv for {len(val_flight_dfs)} launch RPMs')
else:
    print('Validation-subset inputs not found — skipping. Run NB3–NB5 first.')

### 4.7 Validation Checks

In [ ]:
all_ok = True

print('=' * 65)
print('VALIDATION REPORT — NB6 Flight Dynamics')
print('=' * 65)

chk(FLIGHT_DYNAMICS_CSV_PATH.exists(), 'Output CSV exists')
rb = pd.read_csv(FLIGHT_DYNAMICS_CSV_PATH)
chk(rb.shape[0] == len(base_df), f'Row count = {len(base_df)}')
chk(list(flight_df.columns) == OUTPUT_COLUMNS, 'Column schema matches')
chk(flight_df['config_id'].is_unique, 'config_id unique')
chk(flight_df['config_id'].is_monotonic_increasing, 'config_id sorted')

n_ok = int(flight_df['flight_ok'].sum())
n_liftoff = int(flight_df['can_liftoff'].sum())
chk(n_ok > 0, f'At least one successful sim ({n_ok})')
chk(n_liftoff > 0, f'At least one liftoff ({n_liftoff})')

ok = flight_df[flight_df['can_liftoff']]

if not ok.empty:
    chk(ok['h_max_m'].ge(0).all(), 'All liftoff configs: h_max >= 0')
    chk(ok['h_max_m'].ge(1e-6).all(), 'All liftoff configs: h_max >= 1e-6 m  (marginal T/W~1.000 may reach <0.001 mm)')
    chk(ok['flight_time_s'].gt(0).all(), 'All liftoff configs: flight_time > 0')
    chk(ok['T_over_W'].gt(1).all(), 'All liftoff configs: T/W > 1')

    n_tl = int((ok['flight_time_s'] >= TIME_LIMIT_S - 0.01).sum())
    chk(n_tl == 0, f'No liftoff configs hit TIME_LIMIT ({n_tl} hit)')
    chk(ok['h_max_m'].max() < 20.0, f'h_max physically plausible (<20 m), max={ok["h_max_m"].max():.3f} m')
    chk(ok['T_over_W'].max() < 10.0, f'T/W plausible (<10), max={ok["T_over_W"].max():.3f}  (>10 = sentinel mass leak)')
    chk(ok['v_impact_m_s'].lt(0).all(), 'All impact velocities negative (falling)')
    chk(ok['rpm_at_impact'].between(0, SWEEP_RPM_MAX).all(), 'RPM at impact in valid range')
    chk(ok['hover_time_s'].le(ok['flight_time_s']).all(), 'hover_time <= flight_time for all liftoff configs')

    corr = ok[['T_over_W', 'h_max_m']].corr().iloc[0, 1]
    chk(corr > 0.6, f'T/W vs h_max correlation > 0.6 (r={corr:.3f})')

    no_lift = flight_df[flight_df['flight_ok'] & ~flight_df['can_liftoff']]
    chk((no_lift['h_max_m'] == 0.0).all(), 'Non-liftoff configs: h_max = 0')

    print()
    print(f'  Liftoff configs       : {n_liftoff} / {n_ok} with QProp data')
    print(f'  T/W range             : [{ok["T_over_W"].min():.3f}, {ok["T_over_W"].max():.3f}]')
    print(f'  h_max [m]             : mean={ok["h_max_m"].mean():.3f}  median={ok["h_max_m"].median():.3f}  max={ok["h_max_m"].max():.3f}')
    print(f'  flight_time [s]       : mean={ok["flight_time_s"].mean():.2f}   max={ok["flight_time_s"].max():.2f}')
    print(f'  hover_time [s]        : mean={ok["hover_time_s"].mean():.2f}   max={ok["hover_time_s"].max():.2f}')
    print(f'  v_max [m/s]           : mean={ok["v_max_m_s"].mean():.3f}  max={ok["v_max_m_s"].max():.3f}')
    print(f'  v_impact [m/s]        : mean={ok["v_impact_m_s"].mean():.3f}  min={ok["v_impact_m_s"].min():.3f}')
    print(f'  RPM at impact         : mean={ok["rpm_at_impact"].mean():.0f}  min={ok["rpm_at_impact"].min():.0f}')

print()
print('=' * 65)
print('ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED — review above')
print('=' * 65)